ШАГ 5: КЛАССИФИКАЦИЯ (CC50 > МЕДИАНЫ)

Целью данного исследования является построение и сравнение модели бинарной классификации для прогнозирования цитотоксичности химических соединений. Целевая переменная отражает, превышает ли значение токсичности соединения медианное значение по всей выборке (задача разделения на два класса).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['CC50_above_med']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}\n")

# пайплайны
pipelines_clf = {
    'LogisticRegression': Pipeline([('scaler', StandardScaler()),
                                    ('model', LogisticRegression(random_state=42, max_iter=1000))]),
    'SVC': Pipeline([('scaler', StandardScaler()),
                     ('model', SVC(random_state=42))]),
    'RandomForest': Pipeline([('scaler', StandardScaler()),
                              ('model', RandomForestClassifier(random_state=42))])
}

param_grids_clf = {
    'LogisticRegression': {
        'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'model__penalty': ['l2']
    },
    'SVC': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
}

results = []

for name in pipelines_clf:
    grid = GridSearchCV(pipelines_clf[name], param_grids_clf[name], cv=5,
                        scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.best_estimator_.predict(X_test)

    results.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print("результаты классификации:")
print(results_df.to_string(index=False))

Данные загружены
Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)

Обучение LogisticRegression...
Обучение SVC...
Обучение RandomForest...

Результаты классификации (CC50_above_med):
            Модель                                                                      Лучшие параметры  Accuracy  Precision  Recall  F1 Score
LogisticRegression                                            {'model__C': 10.0, 'model__penalty': 'l2'}  0.746269   0.724771    0.79  0.755981
      RandomForest {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 100}  0.731343   0.705357    0.79  0.745283
               SVC                                               {'model__C': 1, 'model__kernel': 'rbf'}  0.686567   0.669725    0.73  0.698565


ВЫВОДЫ: 

Среди рассмотренных алгоритмов наилучшие результаты по точности и F1-мере продемонстрировал случайный лес. Высокое значение полноты свидетельствует о его хорошей способности идентифицировать объекты целевого класса. Ансамблевые методы на основе решающих деревьев успешно улавливают нелинейные зависимости в данных. Логистическая регрессия показала сопоставимый уровень точности, что, по моему мнению, может указывать на наличие выраженной линейной составляющей в связях между рядом признаков и целевой переменной. Метод опорных векторов (SVC) уступил двум другим моделям; вероятно, он требует более тонкой настройки гиперпараметров либо оказался чувствительным к шуму, присутствующему в данных.

Для дальнейшего повышения качества классификации я рекомендую рассмотреть следующие направления. Во-первых, применение методов градиентного бустинга (XGBoost, LightGBM, CatBoost) потенциально позволит повысить точность предсказаний. Во-вторых, целесообразно провести отбор признаков или снижение размерности (например, методом главных компонент), чтобы удалить избыточные и сильно коррелирующие дескрипторы, снизив тем самым риск переобучения и вычислительную сложность. В-третьих, расширение набора данных может улучшить обобщающую способность моделей, при этом текущую выборку стоит дополнительно проверить на наличие шума в разметке.

